# 熱交換器の運転データ（ダミー）の特徴量の探索（LightGBM 単一モデル）

- **作成者**: ImoZukushi
- **作成日**: 2026-10-12
- **目的**: 前処理済みのデータセット（`data/processed/dummy/`）で、特徴量のグループの組み合わせ（パターン）を
  LightGBM で比べ、どの特徴量が効いているかを確かめる。
- **元スクリプト**: `scripts/run_feature_search.py`（同じ処理をCLIで実行できる）
- **前のノートブック**: `004_preprocess_heat_exchanger_dummy.ipynb`（データセットの作成）

## このノートブックでやること

探索の設定 `configs/feature_search/hx_dummy.yaml` に従って、次を行います（仕様は `docs/guides/feature_search.md`）。

1. 設定の読み込みと、パターンごとの実験設定の組み立て（グループ → `data.feature_cols` ・ 特徴量ステップ）
2. 比較の前提の確認（全パターンが同じデータ・CV分割・指標か、使う列がデータにあるか）
3. 全パターンの学習（同じ CV 分割。年ごとに「それより前で学習 → 翌1年で検証」の3 fold）
4. 比較: 基準のパターン（`01_raw`、元データの列だけ）との差、fold ごとのスコア、アブレーション、特徴量の重要度

**前提**: データセットを先に作っておきます（`uv run python scripts/preprocess_heat_exchanger_dummy.py`、
またはノートブック 004）。特徴量の列はデータセットの段階で作られており、探索では列を選ぶだけです。
グループに `features` があるもの（薬剤種類のエンコーディング・圧力損失のラグ）は、CV の fold ごとに Pipeline の中で作られます。

## 1. ライブラリの読み込み

リポジトリの `src/` をimportできるよう、`pyproject.toml` のあるディレクトリ（リポジトリルート）を探して `sys.path` に追加します。

In [ ]:
import sys
import time
from pathlib import Path

import polars as pl
from IPython.display import Image, display

# pyproject.toml を目印にリポジトリルートを探し、src/ をimportパスに追加する
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# src/ 配下のモジュール（上でパスを追加した後にimportする必要がある）
from modeling.config import ExperimentConfig, load_feature_search_config  # noqa: E402
from modeling.experiment import load_table  # noqa: E402
from modeling.feature_search import (  # noqa: E402
    build_patterns,
    run_feature_search,
    validate_patterns,
)
from modeling.tracking import MLflowTracker, NullTracker, Tracker  # noqa: E402

# 表を省略せずに表示する
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(60)

## 2. 実行パラメータ

次のセルは papermill のパラメータセル（`parameters` タグ付き）です。スクリプトの引数に対応します。

| パラメータ | スクリプトの引数 | 既定 |
|---|---|---|
| `CONFIG_PATH` | `--config` | `configs/feature_search/hx_dummy.yaml`（リポジトリルートからの相対パスも可） |
| `PATTERNS` | `--patterns` | `None`（全パターン。基準・アブレーションの土台は自動で含める） |
| `ABLATION` | `--no-ablation` の逆 | `True` |
| `TRACKING` | `--no-tracking` の逆 | `True`（MLflow に記録する） |
| `OUTPUT_ROOT` | `--output-root` | `None`（`outputs/feature_search/`） |

In [ ]:
CONFIG_PATH = "configs/feature_search/hx_dummy.yaml"
PATTERNS = None
ABLATION = True
TRACKING = True
OUTPUT_ROOT = None

In [ ]:
# 相対パスはリポジトリルート基準にする（papermill から文字列で渡された場合も Path に変換する）
config_path = Path(CONFIG_PATH)
if not config_path.is_absolute():
    config_path = REPO_ROOT / config_path
output_root = Path(OUTPUT_ROOT) if OUTPUT_ROOT is not None else None
print(f"探索の設定: {config_path}")
print(f"出力のルート: {output_root or 'outputs/feature_search（既定）'}")

## 3. 設定の読み込みとパターンの組み立て

学習の前に全パターンの実験設定を組み立てて検証します（設定ミスで長い実行が途中で失敗しないように）。
アブレーションのパターン（`<土台>__minus_<グループ>`）は末尾に加わります。

In [ ]:
search = load_feature_search_config(config_path)
patterns = build_patterns(search, only=PATTERNS, ablation=ABLATION)
print(f"特徴量の探索 {search.name}: {len(patterns)}パターン（基準: {search.baseline_pattern}）")

# グループの定義（列とステップ）
pl.DataFrame(
    [
        {
            "group": name,
            "n_columns": len(group.columns),
            "columns": ", ".join(group.columns),
            "steps": ", ".join(s.class_path.rsplit(".", 1)[-1] for s in group.features),
        }
        for name, group in search.groups.items()
    ]
)

In [ ]:
# パターンごとのグループ・列数・Pipeline のステップ数
pl.DataFrame(
    [
        {
            "pattern": p.name,
            "ablated_group": p.ablated_group,
            "n_columns": len(p.config.data.feature_cols or []),
            "n_steps": len(p.config.features),
            "groups": ", ".join(p.groups),
        }
        for p in patterns
    ]
)

### 比較の前提の確認

全パターンが同じデータ・CV分割・指標で評価されること、使う列がすべて学習データにあることを確認します
（スクリプトの `--check` に相当）。

In [ ]:
train_columns = load_table(patterns[0].config.data.train_path).columns
validate_patterns(patterns, train_columns)
first = patterns[0].config
primary = first.primary_metric
print(f"問題ありません（学習データ: {first.data.train_path}、{len(train_columns)}列）")
print(f"CV: {first.cv.method} {first.cv.cutoffs}（終了 {first.cv.valid_end}）、主指標: {primary}")

## 4. 実行

パターンごとに CV 学習します（全19パターンで1分前後）。設定がまったく同じパターンは1回だけ学習し、結果を使い回します。
MLflow には、実験名 `hx_dummy_fe` にパターンごとの run が記録されます（`TRACKING = False` で記録しない）。

In [ ]:
def tracker_factory(cfg: ExperimentConfig) -> Tracker:
    """パターンの実験設定から、実験ログの記録先を作る。"""
    if not TRACKING or not cfg.tracking.enabled:
        return NullTracker()
    return MLflowTracker(
        experiment_name=cfg.tracking.experiment_name or cfg.name,
        tracking_uri=cfg.tracking.tracking_uri,
    )


start = time.perf_counter()
result = run_feature_search(
    search,
    only=PATTERNS,
    ablation=ABLATION,
    tracker_factory=tracker_factory,
    output_root=output_root,
    log=print,
)
print(f"出力: {result.output_dir}")
print(f"経過時間: {time.perf_counter() - start:.1f}秒")

## 5. 結果

### 5.1 パターンの比較

主指標の fold 平均が良い順です。`improvement_<主指標>` は基準より良くなった量（正が良い）、
`fold_wins` は基準に勝った fold の数です。差が `cv_std_<主指標>` より小さい場合は偶然の範囲として扱います。

In [ ]:
result.summary.select(
    "rank",
    "pattern",
    "n_columns",
    "n_model_features",
    f"cv_mean_{primary}",
    f"cv_std_{primary}",
    f"oof_{primary}",
    f"improvement_{primary}",
    "fold_wins",
    "n_folds",
)

In [ ]:
display(Image(filename=str(result.output_dir / "summary.png"), width=1000))

### 5.2 fold ごとのスコア

fold（検証年）によって難しさが違うため、平均だけでなく、fold ごとに順位が入れ替わっていないかを見ます。

In [ ]:
display(Image(filename=str(result.output_dir / "fold_scores.png"), width=1000))
result.fold_scores.filter(pl.col("metric") == primary).pivot(
    on="fold", index="pattern", values="score"
)

### 5.3 アブレーション

土台のパターンからグループを1つずつ外したときに、どれだけ悪くなったか（`loss_when_removed`。正: そのグループは効いている、
負: 外した方が良い）。

In [ ]:
if result.ablation.is_empty():
    print("アブレーションは行っていません")
else:
    display(Image(filename=str(result.output_dir / "ablation.png"), width=800))
    display(result.ablation)

### 5.4 特徴量の重要度

最良のパターンについて、fold モデルの重要度（LightGBM の gain の割合）の平均を上位から表示します。
Pipeline の中で作った列（ラグなど）も含みます。

In [ ]:
best = result.summary["pattern"][0]
print(f"最良のパターン: {best}")
if result.importance.is_empty():
    print("重要度を取得できるモデルではありません")
else:
    display(result.importance.filter(pl.col("pattern") == best).head(20))

In [ ]:
# 主なパターンで、重要度の上位5特徴量を並べて比べる
if not result.importance.is_empty():
    top = (
        result.importance.filter(pl.col("rank") <= 5)
        .with_columns(pl.col("importance_mean").round(3))
        .pivot(on="rank", index="pattern", values="feature")
    )
    display(top)

## 6. まとめと次のステップ

### 結果（`configs/feature_search/hx_dummy.yaml` と現在のデータセットで確認した値。RMSE の fold 平均）
- 元データの列だけ（`01_raw`）の 0.178 に対し、前処理の特徴量を加えると全パターンで全 fold 改善した。
  圧力損失を加えるだけで 0.103、洗浄サイクルまで加えた `05_cleaning_cycle` が 0.088 で最良クラス。
- アブレーションでは、洗浄サイクル（外すと +0.026）・センサーの列・圧力損失が効いている。
  薬剤種類・圧力損失上昇率・原油比率は、外すとわずかに良くなるが、差は fold 間のばらつきより小さい。
- CV の中で作る圧力損失のラグ（`08_all_with_lags`）は、0.096 と全部入り（0.091）より悪かった。

### 注意
- 多くのパターンから最良を選ぶと、その CV スコアは楽観的になる。最終的な評価は探索に使っていない期間で確かめる。
- early stopping を検証 fold で行うため、スコアはわずかに楽観的（全パターン同じ条件なので比較には使える）。

### 次のステップ
- 有望なパターン（`05_cleaning_cycle` など）に `overrides: {explain: {enabled: true}, evaluation: {enabled: true}}` を付けて、
  SHAP と誤差を詳しく確認する。
- 選んだ特徴量でハイパーパラメータを探索する（`scripts/run_pipeline.py`、`docs/guides/hyperparameter_tuning.md`）。
- 新しい特徴量は、前処理スクリプトで列を作ってからグループに加える（系列全体を見るもの）か、
  グループの `features` にステップを書く（学習データから値を学ぶもの・軽い変換）。